# 07 · Grounded NCI PDQ guidance assistant — M3

**Project:** Skin Lesion Triage · BIU DS23 Capstone
**Owner:** Agent hat · **Charter:** M3 (groundedness) and the agent's retrieval tool

## What this notebook does
1. Downloads 7 NCI PDQ Health Professional summaries on skin cancer and saves them as a frozen corpus.
2. Splits them into chunks with stable ids (`pdq:<doc>:sNN:cNN`), embeds them and builds a Chroma index.
3. Answers questions **only** from retrieved chunks, citing their ids, and says "I don't know" otherwise.
4. Measures M3 on 20 frozen questions, before and after one improvement (re-ranking).

> **Why the agent needs it.** The agent's tool `retrieve_guidance_context` reads this same index. Every citation
> in the dermatologist's report is a chunk id from here, and the guardrails check that the quote appears in that
> chunk word for word. Without the corpus the agent still runs, and the report says "No verified guidance found".

> **Runs on its own.** Needs no model and no other notebook. Needs `GEMINI_API_KEY` in Colab Secrets and the
> `agent/` folder in the project folder on Drive.

## Target numbers (from the charter)
| | |
|---|---|
| Frozen evaluation questions | 20 (15 answerable, 5 not in the corpus) |
| Refusals on the 5 out-of-corpus questions | 5 / 5 |
| **Groundedness** (cited ids that were actually retrieved) | **≥ 0.9** |
| Numbers in an answer that appear in the cited chunk | all |

## Outputs
| File | Contents |
|---|---|
| `data/processed/rag/nci_corpus.json` | the frozen NCI PDQ corpus (09_agent_daily_report and 08_agent_m4_vs_ifelse read it) |
| `data/processed/rag/chroma_nci/` | the vector index (rebuilt from the corpus, not committed to Git) |
| `reports/agent/m3_summary.csv` | M3 before / after |
| `reports/agent/m3_answers.csv` | every answer, with its groundedness and number check |

In [ ]:
# Cell 1 · Configuration (identical in every notebook)
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT = "/content/drive/MyDrive/לימודים/Data Science/Skin Lesion Triage for Healthcare Providers"
else:
    PROJECT = os.path.abspath("..")

RAW       = os.path.join(PROJECT, "data", "raw")
PROCESSED = os.path.join(PROJECT, "data", "processed")
FIGURES   = os.path.join(PROJECT, "reports", "figures")

# Where the image archive is stored. Kept as its own setting so the heavy files can later move to a
# shared folder without touching anything else. Colab addresses Drive by path, never by the id in a
# share link, so any override below must be a path under /content/drive/MyDrive/.
IMAGES_DIR = os.path.join(RAW, "images")

for d in (RAW, PROCESSED, FIGURES, IMAGES_DIR):
    os.makedirs(d, exist_ok=True)

Mounted at /content/drive


In [ ]:
# Cell 2 · Agent paths and the API key
# The agent code lives in the project folder on Drive, next to the data: PROJECT/agent/*.py
AGENT_DIR = os.path.join(PROJECT, "agent")                # triage_agent.py, rag.py, m4_eval.py, make_daily_list.py
RAG_DIR   = os.path.join(PROCESSED, "rag")                # nci_corpus.json + the Chroma index
AGENT_OUT = os.path.join(PROJECT, "reports", "agent")     # Word reports, agent logs, M3 / M4 tables
for d in (RAG_DIR, AGENT_OUT):
    os.makedirs(d, exist_ok=True)

os.environ["RAG_DIR"] = RAG_DIR                           # must be set before importing rag / triage_agent
if AGENT_DIR not in sys.path:
    sys.path.insert(0, AGENT_DIR)

def get_secret(name):
    """API keys come only from Colab Secrets (key icon, left bar). A key pasted into a cell is a leaked key."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

assert os.path.exists(os.path.join(AGENT_DIR, "triage_agent.py")), \
    f"agent code not found: upload the agent/ folder to {AGENT_DIR}"
print("agent code :", AGENT_DIR)
print("outputs    :", AGENT_OUT)

agent code : /content/drive/MyDrive/לימודים/Data Science/Skin Lesion Triage for Healthcare Providers/agent
outputs    : /content/drive/MyDrive/לימודים/Data Science/Skin Lesion Triage for Healthcare Providers/reports/agent


In [ ]:
# Cell 3 · Install, seeds
!pip install -q sentence-transformers chromadb beautifulsoup4 requests openai pandas
import json, time, random
import numpy as np, pandas as pd
SEED = 42
random.seed(SEED); np.random.seed(SEED)
import rag

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.


## 1 · The corpus

7 NCI PDQ **Health Professional** summaries: screening, prevention, melanoma treatment, skin cancer treatment,
genetics, childhood melanoma, childhood skin cancer. Downloaded once and frozen in `nci_corpus.json`, so every
later run, and every teammate, retrieves from the same text (`rag.load_or_fetch_corpus(refresh=True)` downloads again).

Licence: PDQ text may be reused as text with credit to NCI; excerpts are not an official PDQ summary.

In [ ]:
# Cell 4 · Download once, then read the frozen copy
corpus = rag.load_or_fetch_corpus()
for key, d in corpus.items():
    print(f"{key:15s} {len(d['sections']):3d} sections  {sum(len(s['text']) for s in d['sections']):>8,} chars  {d['title']}")

screening         9 sections    13,009 chars  Skin Cancer Screening (PDQ) - HP
prevention       31 sections    21,083 chars  Skin Cancer Prevention (PDQ) - HP
melanoma_tx      42 sections   123,426 chars  Melanoma Treatment (PDQ) - HP
skin_tx          40 sections    51,383 chars  Skin Cancer Treatment (PDQ) - HP
genetics         84 sections   213,544 chars  Genetics of Skin Cancer (PDQ) - HP
child_melanoma   10 sections    34,319 chars  Childhood Melanoma Treatment (PDQ) - HP
child_skin        8 sections     9,460 chars  Childhood BCC and SCC of the Skin Treatment (PDQ) - HP


## 2 · The LLM backend

Gemini is reached through its OpenAI-compatible endpoint.
Temperature 0: the same question gives the same answer, which is what makes before / after comparable.
Calls are paced for the free tier, and a call that keeps failing stops the run: a failure must never be
counted as an "I don't know".

In [ ]:
# Cell 5 · chat(): one function for every LLM call in this notebook
BACKEND = "gemini"
MODEL = "gemini-3.5-flash-lite"           # same model as TriageLLM in Studio
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
KEY = get_secret("GEMINI_API_KEY")
MIN_GAP_S = 13                            # free tier: about 4-5 calls a minute
_last = [0.0]

def chat(messages, temperature=0.0, max_tokens=600):
    from openai import OpenAI
    client = OpenAI(api_key=KEY, base_url=BASE_URL)
    for attempt in range(4):
        wait = MIN_GAP_S - (time.time() - _last[0])
        if wait > 0:
            time.sleep(wait)
        _last[0] = time.time()
        try:
            # room for the model's thinking AND the answer with its citation
            r = client.chat.completions.create(model=MODEL, messages=messages,
                                               temperature=temperature, max_tokens=max(max_tokens, 4096))
            text = (r.choices[0].message.content or "").strip()
            if text:
                return text
            print("[empty reply, retrying]")
        except Exception as e:
            print(f"[retry {attempt + 1}] {type(e).__name__}")
            time.sleep(20 * (attempt + 1))
    # A failed call must STOP the run, not be counted as "I don't know"
    raise RuntimeError("LLM unavailable (quota?). Results would be invalid - try again later.")

print("warm-up:", chat([{"role": "user", "content": "Reply with one word: READY"}], max_tokens=20))

warm-up: READY


## 3 · The frozen evaluation set

Written **before** any result is seen, then locked. 15 questions answerable from the corpus (open the source and
confirm the answer is there), 5 that are not.

In [ ]:
# Cell 6 · 20 questions, locked
EVAL = [
    {"q": "What are the risk factors for melanoma?",                          "in_corpus": True},
    {"q": "How accurate is the clinical diagnosis of melanoma?",              "in_corpus": True},
    {"q": "Does skin cancer screening reduce melanoma mortality?",            "in_corpus": True},
    {"q": "What are the harms of skin cancer screening?",                     "in_corpus": True},
    {"q": "Does sunscreen use prevent skin cancer?",                          "in_corpus": True},
    {"q": "How is actinic keratosis treated?",                                "in_corpus": True},
    {"q": "What are the main types of skin cancer?",                          "in_corpus": True},
    {"q": "How is localized basal cell carcinoma treated?",                   "in_corpus": True},
    {"q": "How does melanoma in children differ from melanoma in adults?",    "in_corpus": True},
    {"q": "Which inherited genes increase the risk of melanoma?",             "in_corpus": True},
    {"q": "How is squamous cell carcinoma of the skin treated?",              "in_corpus": True},
    {"q": "What is the role of sentinel lymph node biopsy in melanoma?",      "in_corpus": True},
    {"q": "How is melanoma staged?",                                          "in_corpus": True},
    {"q": "Does indoor tanning increase the risk of skin cancer?",            "in_corpus": True},
    {"q": "What treatments are used for stage IV melanoma?",                  "in_corpus": True},
    {"q": "What is the waiting time for a dermatologist at Clalit?",          "in_corpus": False},
    {"q": "Which smartphone camera takes the best photos of moles?",          "in_corpus": False},
    {"q": "How much does a skin biopsy cost in Israel?",                      "in_corpus": False},
    {"q": "What is the recommended ibuprofen dose after a biopsy?",           "in_corpus": False},
    {"q": "Which health fund app supports photo uploads?",                    "in_corpus": False},
]
assert len(EVAL) == 20, "write exactly 20 questions"
assert sum(not e["in_corpus"] for e in EVAL) == 5, "exactly 5 must be out of the corpus"
print("eval set locked:", len(EVAL))

eval set locked: 20


## 4 · The metric

**Groundedness** = the share of chunk ids cited in an answer that were actually retrieved for that question
(Module 5, with chunk ids instead of arXiv ids). An answer that cites nothing scores 0. The cell checks the
metric on three hand-made cases before it is trusted.

In [ ]:
# Cell 7 · Verify the metric before using it
assert rag.groundedness("See [pdq:screening:s04:c02].", ["pdq:screening:s04:c02"]) == 1.0
assert rag.groundedness("See [pdq:screening:s04:c02] and [pdq:genetics:s01:c00].", ["pdq:screening:s04:c02"]) == 0.5
assert rag.groundedness("No citation here.", ["pdq:screening:s04:c02"]) == 0.0
print("groundedness metric verified")

groundedness metric verified


## 5 · Chunk, embed, index

PDQ sections run to thousands of characters, so a whole section would mix several ideas in one vector.
Chunks of ~900 characters with ~150 overlap, cut on sentence boundaries, keep about one idea per chunk; the
overlap keeps a sentence that falls on a boundary. Embedding: `all-MiniLM-L6-v2`, normalized.

In [ ]:
# Cell 8 · Build the index (about a minute)
rag.build_index()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/23 [00:00<?, ?it/s]

indexed 724 chunks from 7 documents


724

In [ ]:
# Cell 9 · One question end to end: the answer and the ids it was allowed to cite
ans, ids = rag.answer(EVAL[0]["q"], chat, k=3)
print(ans, "\n\nretrieved:", ids)

Risk factors for melanoma include both intrinsic (genetic and phenotype) and extrinsic (environmental or exposure) factors, which include pigmentary characteristics, as well as a family and personal history of melanoma [pdq:melanoma_tx:s02:c00]. 

retrieved: ['pdq:melanoma_tx:s02:c00', 'pdq:genetics:s47:c00', 'pdq:genetics:s69:c01']


## 6 · The "I don't know" test

The 5 out-of-corpus questions must be refused. An answer here means the model used its own knowledge,
which is exactly what a grounded assistant may not do.

In [ ]:
# Cell 10 · Refusals on the 5 out-of-corpus questions
out_of_scope = [e for e in EVAL if not e["in_corpus"]]
refused = 0
for e in out_of_scope:
    a, _ = rag.answer(e["q"], chat, k=3)
    refused += rag.is_refusal(a)
    if not rag.is_refusal(a):
        print("ANSWERED ANYWAY:", e["q"], "->", a[:150])
    time.sleep(4)
print(f"refused {refused} of {len(out_of_scope)}   (target 5 of 5)")

refused 5 of 5   (target 5 of 5)


## 7 · Before: the baseline on the 15 answerable questions

In [ ]:
# Cell 11 · Baseline: groundedness, numbers found in the cited chunk, wrong refusals
def evaluate(**kw):
    rows = []
    for e in EVAL:
        if not e["in_corpus"]:
            continue
        a, ids = rag.answer(e["q"], chat, k=3, **kw)
        rows.append({"q": e["q"], "groundedness": rag.groundedness(a, ids),
                     "numbers_ok": rag.numbers_supported(a), "refused": rag.is_refusal(a), "answer": a})
        time.sleep(4)
    return pd.DataFrame(rows)

before = evaluate()
print("groundedness:", round(before.groundedness.mean(), 3), "| numbers in cited chunk:",
      f"{before.numbers_ok.sum()}/{len(before)}", "| wrongly refused:", before.refused.sum())

groundedness: 0.733 | numbers in cited chunk: 15/15 | wrongly refused: 4


## 8 · One improvement: re-ranking

Retrieve wide (20 chunks), then re-rank with a cross-encoder (`ms-marco-MiniLM-L-6-v2`) and keep 3.
PDQ treatment sections are long and alike, so the embedding's top 3 often include near-misses; a cross-encoder
reads the question and the chunk together. **Only this changes**: questions, prompt, k and model stay the same.

In [ ]:
# Cell 12 · After: the same evaluation with re-ranking
after = evaluate(rerank=True)
print(f"before {before.groundedness.mean():.3f} | after {after.groundedness.mean():.3f} | "
      f"lift {after.groundedness.mean() - before.groundedness.mean():+.3f}   (target >= 0.9)")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

before 0.733 | after 0.933 | lift +0.200   (target >= 0.9)


**Interpretation.** Re-ranking lifted groundedness from 0.733 to 0.933 (+0.200), above the 0.9 target.
Before, 4 of the 15 answerable questions were refused: the embedding's top 3 missed the passage that held the
answer, so the assistant said "I don't know" rather than invent, which is the safe failure. The cross-encoder
brought the right passage into the top 3. All 5 out-of-corpus questions were refused, and every number matched
its cited chunk.

**Run-to-run variation.** An earlier run of this notebook gave 0.600 before (6 refused) and the same 0.933 after.
The model is called at temperature 0 but its answers still vary slightly between runs; the baseline moves, the
re-ranked score held at 0.933 in both runs.

**Limits.** 15 answerable questions is a small set (one question = 0.067). Groundedness checks that every cited
id was retrieved, not that the answer is clinically complete; the number check covers numbers only.

**Decision:** keep re-ranking; it is now used by the agent's `retrieve_guidance_context` tool.

In [ ]:
# Cell 13 · Save the results
summary = pd.DataFrame([
    {"run": "before", "groundedness": before.groundedness.mean(), "numbers_ok": before.numbers_ok.mean(), "refusals": f"{refused}/5"},
    {"run": "after (rerank)", "groundedness": after.groundedness.mean(), "numbers_ok": after.numbers_ok.mean(), "refusals": f"{refused}/5"},
])
summary.to_csv(os.path.join(AGENT_OUT, "m3_summary.csv"), index=False)
pd.concat([before.assign(run="before"), after.assign(run="after")]).to_csv(os.path.join(AGENT_OUT, "m3_answers.csv"), index=False)
print("saved to", AGENT_OUT)
summary

saved to /content/drive/MyDrive/לימודים/Data Science/Skin Lesion Triage for Healthcare Providers/reports/agent


,run,groundedness,numbers_ok,refusals
0,before,0.733333,1.0,5/5
1,after (rerank),0.933333,1.0,5/5


## Summary

### Corpus
| Check | Expected | Actual |
|---|---|---|
| NCI PDQ documents | 7 | **7** ✓ |
| Sections / chunks | — | 224 sections · **724 chunks** |

### M3
| Check | Expected | Actual |
|---|---|---|
| Frozen questions | 20 (15 + 5) | **20** ✓ |
| Refusals on out-of-corpus questions | 5 / 5 | **5 / 5** ✓ |
| Groundedness, before | — | 0.733 (4 of 15 answerable questions refused) · earlier run 0.600 |
| **Groundedness, after re-ranking** | **≥ 0.9** | **0.933** ✓ (same in both runs) |
| Answers whose numbers all appear in the cited chunk | 15 / 15 | **15 / 15** ✓ |

### Decisions (Module 5 template: what, why, the rejected alternative, evidence)
| Decision | Choice | Why | Rejected alternative | Evidence |
|---|---|---|---|---|
| Corpus | 7 NCI PDQ Health Professional summaries | authoritative, free to reuse as text, written for clinicians | general web search | frozen in `nci_corpus.json` |
| Chunking | ~900 chars, 150 overlap, sentence boundaries | PDQ sections are long; one idea per chunk | whole sections | 724 chunks from 224 sections |
| Evaluation | 15 answerable + 5 out-of-corpus questions, locked before any result | measures both grounded answers and refusals | answerable questions only | 5 / 5 refusals |
| Improvement | cross-encoder re-ranking (wide 20 → keep 3) | PDQ sections are long and alike; the embedding's top 3 had near-misses | larger k | groundedness 0.733 → 0.933 |

### What the agent uses from this notebook
- The frozen corpus and the same chunk ids: every citation in the dermatologist's report is one of these ids.
- Re-ranking, kept here, is switched on in the agent's retrieval tool.
- Adults (18+) are never shown the two childhood pages; the query is built by code from the symptoms and body site.

**Next:** `08_agent_m4_vs_ifelse.ipynb` — is the agent needed at all?